In [ ]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# ============================================================
# TASK 5: Explainability (Grad-CAM / Grad-CAM++) + Generalizability
# ============================================================

import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# -----------------------------
# CONFIG
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (300, 300)
BATCH_SIZE = 32
EPOCHS = 10
LAST_CONV_LAYER_NAME = "top_conv"
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]

# =========================
# CORRECT REAL PATHS
# =========================
BASE_DIR = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
SECOND_BASE_DIR = "/kaggle/input/datasets/prashant268/chest-xray-covid19-pneumonia/Data"

# If you already saved your best model, put path here; else keep None
MODEL_PATH = None

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

SECOND_TRAIN_DIR = os.path.join(SECOND_BASE_DIR, "train")
SECOND_TEST_DIR  = os.path.join(SECOND_BASE_DIR, "test")

print("Original dataset path:", BASE_DIR)
print("Second dataset path  :", SECOND_BASE_DIR)

# -----------------------------
# PATH CHECK
# -----------------------------
required_paths = [TRAIN_DIR, VAL_DIR, TEST_DIR, SECOND_TRAIN_DIR, SECOND_TEST_DIR]
for p in required_paths:
    print(p, "->", os.path.exists(p))

# -----------------------------
# DATA GENERATORS
# -----------------------------
train_datagen = ImageDataGenerator(
    preprocessing_function=eff_preprocess,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

eval_datagen = ImageDataGenerator(
    preprocessing_function=eff_preprocess
)

# -----------------------------
# ORIGINAL DATASET
# -----------------------------
train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True,
    seed=SEED
)

val_data = eval_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

test_data = eval_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_data.classes),
    y=train_data.classes
)
class_weights = dict(enumerate(class_weights))

print("Original dataset class weights:", class_weights)
print("Original class indices:", train_data.class_indices)

# -----------------------------
# MODEL
# -----------------------------
def build_efficientnetb3_model(fine_tune_layers=50):
    base_model = EfficientNetB3(
        weights="imagenet",
        include_top=False,
        input_shape=(300, 300, 3)
    )

    for layer in base_model.layers[:-fine_tune_layers]:
        layer.trainable = False
    for layer in base_model.layers[-fine_tune_layers:]:
        layer.trainable = True

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs=base_model.input, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# -----------------------------
# LOAD OR TRAIN BEST MODEL
# -----------------------------
if MODEL_PATH is not None and os.path.exists(MODEL_PATH):
    best_model = tf.keras.models.load_model(MODEL_PATH)
    print("Loaded saved model:", MODEL_PATH)
else:
    print("No saved model found. Training EfficientNetB3 on original dataset...")
    best_model = build_efficientnetb3_model(fine_tune_layers=50)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            patience=2,
            factor=0.3
        )
    ]

    start_train_time = time.time()
    history = best_model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )
    original_train_time = time.time() - start_train_time
    print(f"Original dataset training time: {original_train_time:.2f} sec")

# -----------------------------
# EVALUATION HELPERS
# -----------------------------
def evaluate_binary_model(model, generator, threshold=0.5):
    generator.reset()
    start_time = time.time()

    probs = model.predict(generator, verbose=1).flatten()
    preds = (probs > threshold).astype(int)
    y_true = generator.classes

    test_time = time.time() - start_time

    acc = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    auc = roc_auc_score(y_true, probs)

    cm = confusion_matrix(y_true, preds)
    class_acc = cm.diagonal() / cm.sum(axis=1)
    fpr, tpr, _ = roc_curve(y_true, probs)

    return {
        "y_true": y_true,
        "y_pred": preds,
        "y_prob": probs,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc,
        "confusion_matrix": cm,
        "class_accuracy": class_acc,
        "fpr": fpr,
        "tpr": tpr,
        "test_time": test_time
    }

def evaluate_dataframe_generator(model, generator, y_true, threshold=0.5):
    generator.reset()
    start_time = time.time()

    probs = model.predict(generator, verbose=1).flatten()
    preds = (probs > threshold).astype(int)

    test_time = time.time() - start_time

    acc = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    auc = roc_auc_score(y_true, probs)

    cm = confusion_matrix(y_true, preds)
    class_acc = cm.diagonal() / cm.sum(axis=1)
    fpr, tpr, _ = roc_curve(y_true, probs)

    return {
        "y_true": y_true,
        "y_pred": preds,
        "y_prob": probs,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc,
        "confusion_matrix": cm,
        "class_accuracy": class_acc,
        "fpr": fpr,
        "tpr": tpr,
        "test_time": test_time
    }

# -----------------------------
# ORIGINAL DATASET EVALUATION
# -----------------------------
results_original = evaluate_binary_model(best_model, test_data)

print("\n===== ORIGINAL DATASET TEST RESULTS =====")
print(f"Accuracy      : {results_original['accuracy']:.4f}")
print(f"Precision     : {results_original['precision']:.4f}")
print(f"Recall        : {results_original['recall']:.4f}")
print(f"F1-Score      : {results_original['f1']:.4f}")
print(f"ROC-AUC       : {results_original['auc']:.4f}")
print(f"Testing Time  : {results_original['test_time']:.2f} sec")

print("\nClassification Report:")
print(classification_report(
    results_original["y_true"],
    results_original["y_pred"],
    target_names=CLASS_NAMES,
    digits=4
))

print("\nPer-class accuracy:")
for i, acc in enumerate(results_original["class_accuracy"]):
    print(f"{CLASS_NAMES[i]}: {acc:.4f}")

disp = ConfusionMatrixDisplay(
    confusion_matrix=results_original["confusion_matrix"],
    display_labels=CLASS_NAMES
)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Original Dataset Confusion Matrix")
plt.show()

plt.figure(figsize=(6, 5))
plt.plot(results_original["fpr"], results_original["tpr"], label=f"AUC = {results_original['auc']:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Original Dataset ROC Curve")
plt.legend(loc="lower right")
plt.show()

# -----------------------------
# GRAD-CAM HELPERS
# -----------------------------
def load_and_preprocess_image(img_path, img_size=IMG_SIZE):
    img = load_img(img_path, target_size=img_size)
    arr = img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = eff_preprocess(arr.copy())
    return arr

def get_display_image(img_path, img_size=IMG_SIZE):
    img = load_img(img_path, target_size=img_size)
    return img_to_array(img).astype("uint8")

def make_gradcam_heatmap(img_array, model, last_conv_layer_name=LAST_CONV_LAYER_NAME):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_channel = predictions[:, 0]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def make_gradcam_plus_plus_heatmap(img_array, model, last_conv_layer_name=LAST_CONV_LAYER_NAME):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    img_tensor = tf.cast(img_array, tf.float32)

    with tf.GradientTape(persistent=True) as tape1:
        with tf.GradientTape(persistent=True) as tape2:
            conv_outputs, predictions = grad_model(img_tensor)
            class_channel = predictions[:, 0]

        first_grads = tape2.gradient(class_channel, conv_outputs)
    second_grads = tape1.gradient(first_grads, conv_outputs)

    conv_outputs = conv_outputs[0]
    first_grads = first_grads[0]
    second_grads = second_grads[0]

    alpha_num = second_grads
    alpha_denom = 2.0 * second_grads
    alpha_denom = tf.where(alpha_denom != 0.0, alpha_denom, tf.ones_like(alpha_denom))

    alphas = alpha_num / (alpha_denom + 1e-8)
    weights = tf.reduce_sum(tf.nn.relu(first_grads) * alphas, axis=(0, 1))

    cam = tf.reduce_sum(weights * conv_outputs, axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)

    del tape1
    del tape2
    return cam.numpy()

def overlay_heatmap_on_image(original_img, heatmap, alpha=0.4):
    heatmap = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed = cv2.addWeighted(original_img, 1 - alpha, heatmap, alpha, 0)
    return superimposed

# -----------------------------
# GRAD-CAM VISUALIZATION
# -----------------------------
filepaths = test_data.filepaths
y_true = results_original["y_true"]
y_pred = results_original["y_pred"]
y_prob = results_original["y_prob"]

correct_indices = np.where(y_true == y_pred)[0]
incorrect_indices = np.where(y_true != y_pred)[0]

print("\nCorrect samples available  :", len(correct_indices))
print("Incorrect samples available:", len(incorrect_indices))

num_correct = min(3, len(correct_indices))
num_incorrect = min(3, len(incorrect_indices))

selected_correct = correct_indices[:num_correct]
selected_incorrect = incorrect_indices[:num_incorrect]
selected_indices = list(selected_correct) + list(selected_incorrect)

SAVE_DIR = "/kaggle/working/task5_gradcam_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

for idx in selected_indices:
    img_path = filepaths[idx]
    true_label = int(y_true[idx])
    pred_label = int(y_pred[idx])
    prob = float(y_prob[idx])

    img_array = load_and_preprocess_image(img_path)
    original_img = get_display_image(img_path)

    gradcam_heatmap = make_gradcam_heatmap(img_array, best_model)
    gradcampp_heatmap = make_gradcam_plus_plus_heatmap(img_array, best_model)

    gradcam_overlay = overlay_heatmap_on_image(original_img, gradcam_heatmap, alpha=0.4)
    gradcampp_overlay = overlay_heatmap_on_image(original_img, gradcampp_heatmap, alpha=0.4)

    status = "CORRECT" if true_label == pred_label else "INCORRECT"

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(original_img.astype("uint8"))
    plt.title(f"Original\nTrue: {CLASS_NAMES[true_label]} | Pred: {CLASS_NAMES[pred_label]}\nProb={prob:.4f} | {status}")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(cv2.cvtColor(gradcam_overlay, cv2.COLOR_BGR2RGB))
    plt.title("Grad-CAM")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(cv2.cvtColor(gradcampp_overlay, cv2.COLOR_BGR2RGB))
    plt.title("Grad-CAM++")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    file_name = os.path.basename(img_path)
    save_status = "correct" if true_label == pred_label else "incorrect"
    cv2.imwrite(os.path.join(SAVE_DIR, f"{save_status}_gradcam_{file_name}"), gradcam_overlay)
    cv2.imwrite(os.path.join(SAVE_DIR, f"{save_status}_gradcampp_{file_name}"), gradcampp_overlay)

print("Saved Grad-CAM outputs to:", SAVE_DIR)

# -----------------------------
# SECOND DATASET PREPARATION
# Merge labels:
# NORMAL -> 0
# COVID19 / PNEUMONIA -> 1
# -----------------------------
def build_second_dataset_df(directory):
    filepaths = []
    labels = []

    for label_name in sorted(os.listdir(directory)):
        label_path = os.path.join(directory, label_name)

        if not os.path.isdir(label_path):
            continue

        for fname in os.listdir(label_path):
            fpath = os.path.join(label_path, fname)
            if os.path.isfile(fpath):
                filepaths.append(fpath)
                if label_name.upper() == "NORMAL":
                    labels.append(0)
                else:
                    labels.append(1)

    return pd.DataFrame({"filename": filepaths, "class": labels})

second_train_full_df = build_second_dataset_df(SECOND_TRAIN_DIR)
second_test_df = build_second_dataset_df(SECOND_TEST_DIR)

print("\nSecond train full shape:", second_train_full_df.shape)
print("Second test shape      :", second_test_df.shape)
print("\nSecond train class distribution:")
print(second_train_full_df["class"].value_counts())
print("\nSecond test class distribution:")
print(second_test_df["class"].value_counts())

second_train_df, second_val_df = train_test_split(
    second_train_full_df,
    test_size=0.2,
    random_state=SEED,
    stratify=second_train_full_df["class"]
)

second_train_df = second_train_df.reset_index(drop=True)
second_val_df = second_val_df.reset_index(drop=True)
second_test_df = second_test_df.reset_index(drop=True)

second_train_gen = train_datagen.flow_from_dataframe(
    second_train_df,
    x_col="filename",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=True,
    seed=SEED
)

second_val_gen = eval_datagen.flow_from_dataframe(
    second_val_df,
    x_col="filename",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False
)

second_test_gen = eval_datagen.flow_from_dataframe(
    second_test_df,
    x_col="filename",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False
)

second_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(second_train_df["class"]),
    y=second_train_df["class"]
)
second_class_weights = dict(enumerate(second_class_weights))
print("\nSecond dataset class weights:", second_class_weights)

# -----------------------------
# TRAIN ON SECOND DATASET
# -----------------------------
general_model = build_efficientnetb3_model(fine_tune_layers=50)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        patience=2,
        factor=0.3
    )
]

start_general_train = time.time()
history_general = general_model.fit(
    second_train_gen,
    validation_data=second_val_gen,
    epochs=EPOCHS,
    class_weight=second_class_weights,
    callbacks=callbacks,
    verbose=1
)
general_train_time = time.time() - start_general_train
print(f"Second dataset training time: {general_train_time:.2f} sec")

# -----------------------------
# SECOND DATASET EVALUATION
# -----------------------------
second_y_true = second_test_df["class"].values

results_second = evaluate_dataframe_generator(
    general_model,
    second_test_gen,
    second_y_true
)

print("\n===== SECOND DATASET TEST RESULTS =====")
print(f"Accuracy      : {results_second['accuracy']:.4f}")
print(f"Precision     : {results_second['precision']:.4f}")
print(f"Recall        : {results_second['recall']:.4f}")
print(f"F1-Score      : {results_second['f1']:.4f}")
print(f"ROC-AUC       : {results_second['auc']:.4f}")
print(f"Testing Time  : {results_second['test_time']:.2f} sec")

print("\nClassification Report:")
print(classification_report(
    results_second["y_true"],
    results_second["y_pred"],
    target_names=CLASS_NAMES,
    digits=4
))

print("\nPer-class accuracy:")
for i, acc in enumerate(results_second["class_accuracy"]):
    print(f"{CLASS_NAMES[i]}: {acc:.4f}")

disp = ConfusionMatrixDisplay(
    confusion_matrix=results_second["confusion_matrix"],
    display_labels=CLASS_NAMES
)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Second Dataset Confusion Matrix")
plt.show()

plt.figure(figsize=(6, 5))
plt.plot(results_second["fpr"], results_second["tpr"], label=f"AUC = {results_second['auc']:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Second Dataset ROC Curve")
plt.legend(loc="lower right")
plt.show()

# -----------------------------
# COMPARISON
# -----------------------------
comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Original Dataset": [
        results_original["accuracy"],
        results_original["precision"],
        results_original["recall"],
        results_original["f1"],
        results_original["auc"]
    ],
    "Second Dataset": [
        results_second["accuracy"],
        results_second["precision"],
        results_second["recall"],
        results_second["f1"],
        results_second["auc"]
    ]
})

comparison_df["Difference (Second - Original)"] = (
    comparison_df["Second Dataset"] - comparison_df["Original Dataset"]
)

print("\n===== GENERALIZABILITY COMPARISON =====")
print(comparison_df)

comparison_csv_path = "/kaggle/working/task5_generalizability_comparison.csv"
comparison_df.to_csv(comparison_csv_path, index=False)
print("\nSaved comparison CSV to:", comparison_csv_path)

# -----------------------------
# SAVE TASK 5 ANALYSIS TEXT
# -----------------------------
analysis_text = f"""
Task 5 Analysis

The best-performing model used for explainability was EfficientNetB3. Grad-CAM and Grad-CAM++
were applied on correctly classified and incorrectly classified chest X-ray images from the original
dataset. For correct predictions, the model generally focused on important lung regions. For
misclassified images, attention was often weak, diffuse, or shifted toward less relevant regions.

For generalizability testing, the same EfficientNetB3 architecture was retrained on a second
chest X-ray dataset from the same medical imaging domain. In this second dataset, the labels
were converted into binary classes by mapping NORMAL to class 0 and merging COVID19 and
PNEUMONIA into class 1.

The comparison between the original dataset and second dataset shows how well the model
generalizes under dataset shift. A noticeable drop in performance indicates limited robustness and
suggests that the model learned dataset-specific patterns rather than fully general disease features.
"""

print("\n" + analysis_text)

analysis_path = "/kaggle/working/task5_analysis.txt"
with open(analysis_path, "w") as f:
    f.write(analysis_text)
print("Saved analysis text to:", analysis_path)

# -----------------------------
# SAVE MODELS
# -----------------------------
best_model_path = "/kaggle/working/task5_best_model_original.keras"
general_model_path = "/kaggle/working/task5_general_model_second_dataset.keras"

best_model.save(best_model_path)
general_model.save(general_model_path)

print("Saved original model to:", best_model_path)
print("Saved second-dataset model to:", general_model_path)
print("\nTask 5 complete.")